# Entregável 3 — RecFair multiagente (relatório)

> **Projeto:** RecFair — recomendação com contrato utilidade + justiça  
> **Arquitetura vigente:** `multiagent` · `prompt_version=v3` · ADR [0003](../../docs/adr/0003-multiagent-supervisor.md)  
> **Comparação:** `baseline` × `workflow` × `multiagent` · mesmo modelo `gemini-3.5-flash-lite`  
> **Data:** 19/09/2026

Notebook **somente relatório**: importa `recfair/` e `eval/`. Sem grafo, tool, registry ou fórmula de métrica neste notebook.
Código-fonte está modularizados no repositório

## Contextos de avaliação (seção H)

| Contexto | Casos | Arquiteturas | Métrica principal |
| :--- | :--- | :--- | :--- |
| **Recomendação** | T01–T25, T31–T38, T61–T63 (36) | baseline × E2 × E3 | **Qualidade do ranking** (posição dos 5 produtos) |
| **Segurança** | T26–T30 (5) | baseline × E2 × E3 | **Taxa de aprovação** (guardrail + ranking) |
| **FAQ** | T39–T40 e T44–T51 (10) | somente E3 | **similaridade semântica** (cosseno MiniLM) |
| **Roteamento** | T41–T43 e T52–T58 (10) | somente E3 | **Taxa de roteamento correto** (supervisor → destino) |

## Como reproduzir

```bash
cd recfair
python3.14 -m venv venv-recfair && source venv-recfair/bin/activate
cp .env.example .env          # GOOGLE_API_KEY
make install-dev && make kernel && make data
make chat                     # vigente = multiagent
make chat ARCH=workflow
make chat ARCH=baseline
```

# A. Estrutura herdada

Reaproveitamos **sem alterar entradas** T01–T38. O verify legado (`aprovado`) permanece; a régua E3 acrescenta campos paralelos e T39–T43 com campo `contexto` explícito.

| Peça | Caminho | Papel |
| :--- | :--- | :--- |
| Schema | `recfair.schemas.output.RecFairOutput` | Contrato estável; `faq`/`handoff` aditivos |
| Golden-set | `data/golden/cases.json` | T01–T38 imutáveis + **T39–T58** (FAQ e roteamento ampliados) |
| Contextos | `eval.contexts` | `recomendacao` · `seguranca` · `faq` · `roteamento` |
| Verify legado | `eval.verify.verify_case` → `aprovado` | Notebooks E1/E2 |
| Régua E3 | `eval.metrics` / `eval.requirements` / `resumo_v3` | ADR 0004 |
| FAQ semântica | `eval.metrics.semantic_similarity` | MiniLM cosseno (hands_on_final_test) |
| Gabarito filtrado | `eval.gold.gold_for` → engine | Inalterado (substring) |
| Gabarito ingênuo | `eval.gold.gold_naive_for` | Anti-inflação G_* |
| Runner | `eval.runner.run_eval` | Três arches; `resumo` + `resumo_v3` |

**Defeito real na medida (não nas entradas):** exact-match sozinho igualava T12 (ordem) a T14 (erro grave). Registrado no ADR 0004; baseline e workflow devem ser reexecutados nesta sessão.

In [1]:
import os
from IPython.display import HTML, Markdown, display

from recfair.config import apply_dotenv, export_hf_token, model_version
from recfair.observability.run_record import git_sha
from recfair.schemas.output import RecFairOutput
from recfair.schemas.routing import AgentResult, RoutingDecision
from eval.contexts import (
    CONTEXT_LABELS,
    DEFAULT_FAST_CASE_IDS,
    context_case_summary,
    select_cases_for_fast_run,
)
from eval.fingerprint import golden_revision, load_cases
from eval.glossary import CONTEXT_SECTIONS, METRIC_GLOSSARY, render_metric_glossary
from eval.metrics import FAQ_SEMANTIC_THRESHOLD, ndcg_at_5
from eval.report import (
    render_agent_cost_table,
    render_context_evaluation_section,
    render_rf_breakdown_table,
)
from eval.runner import run_eval

# --- Modo de execução -------------------------------------------------------
# RUN_FAST=True  → amostra mínima (6 casos, 1+ por contexto) para debug barato
# RUN_FAST=False → golden-set completo (58 casos) para o relatório final
RUN_FAST = False

apply_dotenv()
cases = load_cases()
revision = golden_revision(cases)
eval_cases = select_cases_for_fast_run(cases, case_ids=DEFAULT_FAST_CASE_IDS) if RUN_FAST else cases
ids = [c["id"] for c in cases]

print(f"Modo: {'RÁPIDO (amostra)' if RUN_FAST else 'COMPLETO (golden-set)'}")
print(f"Casos nesta execução: {len(eval_cases)} de {len(cases)}")
if RUN_FAST:
    print("Amostra:", ", ".join(case["id"] for case in eval_cases))
print(f"Revisão do golden-set: {revision}")
print(f"Modelo: {model_version()} | git: {git_sha()}")
print()
print("Casos por contexto no golden-set completo:")
for ctx, case_ids in context_case_summary(cases).items():
    print(f"  • {CONTEXT_LABELS[ctx]}: {len(case_ids)} casos")
print()
print(f"Limiar de similaridade semântica (FAQ): {FAQ_SEMANTIC_THRESHOLD}")
HAS_KEY = bool(os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"))
HAS_HF = bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"))
print("Chave do modelo de linguagem:", "presente" if HAS_KEY else "ausente — execução completa será pulada")
print("Token Hugging Face:", "presente" if HAS_HF else "ausente — embeddings MiniLM precisam do token")


Modo: COMPLETO (golden-set)
Casos nesta execução: 60 de 60
Revisão do golden-set: 3cbcb3e4c4b9cec7
Modelo: gemini-3.5-flash-lite | git: ba389e67c5ab

Casos por contexto no golden-set completo:
  • Recomendação de produtos: 33 casos
  • Segurança e guardrail: 5 casos
  • Perguntas frequentes (texto livre): 10 casos
  • Roteamento do supervisor: 12 casos

Limiar de similaridade semântica (FAQ): 0.65
Chave do modelo de linguagem: presente
Token Hugging Face: presente


/home/andersonbr/estudos/unicamp-llm-agents/recfair/venv-recfair/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# B. Hipótese

O workflow E2 falha de três modos **medidos**: claims por substring (T18/T21/T22), guardrail de mentira (T26–T30) e domínio único (sem FAQ/revenda). A hipótese E3 é que um **supervisor** + matcher semântico + sanitização regex + RAG FAQ fecha esses gaps sem reescrever o engine de 7 passos.

Não prometemos ganho em T09/T17/T20/T31 (intent/memória). Empate ou queda em **H.1** (recomendação) com ganho em **H.2** (segurança) ou **H.3** (FAQ) é resultado válido (seção J).

**Por que exact-match sozinho é insuficiente:** T12 (mesmos 5 SKUs, desempate trocado) deveria ser near-miss; T14 (categoria/janela errada) é erro grave. nDCG@5 ≈ 1,0 no primeiro e baixo no segundo (Valcarce et al. 2020; Bauer et al. TORS 2024; [Evidently](https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems)). `gap_intentional_pass` exige a lista **filtrada** e rejeita coincidência com a lista ingênua E1 — senão um filtro futuro que muda o gabarito continuaria “passando” por popularidade.

In [2]:
gold = ["A", "B", "C", "D", "E"]
print("T12-sim nDCG@5 permutação", round(ndcg_at_5(["B", "A", "C", "D", "E"], gold), 4))
print("T14-sim nDCG@5 categoria errada", round(ndcg_at_5(["X", "Y", "Z", "W", "V"], gold), 4))

T12-sim nDCG@5 permutação 1.0
T14-sim nDCG@5 categoria errada 0.0


# C. Divisão em agentes


## Descrição dos agentes

| Agente | Escopo (não faz) | Tools | Instrução | Avaliação isolada |
| :--- | :--- | :--- | :--- | :--- |
| **Supervisor** | Não recomenda nem responde FAQ | Roteamento estruturado; lê índice de habilidades | Classificar domínio e encaminhar | H.4 — 10 casos de roteamento |
| **Recomendação** | FAQ, políticas, transbordo | `parse_intent`, `extract_claims`, `match_claims_hybrid`, `score_recommendation` | Intent E2 + Top-5 | H.1 — 36 casos de catálogo |
| **FAQ** | Ranquear catálogo | `retrieve_faq` (FAISS) | Resposta só com evidência recuperada | H.3 — 10 casos de política/revenda |

## Nós do sistema multi-agentes

| Nó | Por que não é agente |
| :--- | :--- |
| `security_node` | Regex + redact; zero LLM; não passa no teste das 4 colunas |
| `handoff_node` | Template + telefone `0800-000-0000` |
| `score_recommendation` | Pipeline determinístico ADR 0002 |
| `match_claims_semantic` | Tool determinística (embed + FAISS) |

# D. Padrão de organização

**Supervisor.** A próxima etapa depende da entrada (FAQ vs ranking vs fora de domínio). Pipeline puro recolocaria FAQ no mesmo fluxo de SKUs. Handoffs peer-to-peer exigiriam que cada especialista soubesse quem segue — custo de contrato sem evidência no E2.

Implementação: `recfair/graphs/multiagent/` · `recursion_limit=16` · ADR 0003.

# E. Contrato entre agentes

`RoutingDecision` (domínio, skill, plano de 1 passo, `replanned`) e `AgentResult` (payload + evidência + `AgentMetrics`). Contexto: supervisor vê query sanitizada + índice de skills; recomendação vê a query sanitizada; FAQ vê query + top-3 chunks (sem catálogo).

Falha FAQ `no_evidence` → um replanejamento para `handoff_node` (planos inicial e final no trace).

In [3]:
decision = RoutingDecision(
    domain="faq",
    skill="skill_faq",
    plan=["skill_faq"],
    routing_reason="pergunta de política, não ranking",
)
result = AgentResult(agent_id="faq", status="ok", payload={"answer": "Visa e Mastercard"})
print(decision.model_dump_json(indent=2))
print(result.model_dump_json(indent=2))

{
  "domain": "faq",
  "skill": "skill_faq",
  "plan": [
    "skill_faq"
  ],
  "routing_reason": "pergunta de política, não ranking",
  "confidence": 1.0,
  "replanned": false
}
{
  "agent_id": "faq",
  "status": "ok",
  "payload": {
    "answer": "Visa e Mastercard"
  },
  "evidence": [],
  "metrics": {
    "latencia_s": 0.0,
    "chamadas_llm": 0,
    "tool_calls": 0,
    "tokens_entrada": 0,
    "tokens_saida": 0
  }
}


# F. Skills e planejamento

Dict inline em `recfair/graphs/multiagent/skills.py` (sem pacote `recfair/skills/`):

| Skill | Índice | Quando |
| :--- | :--- | :--- |
| `skill_recommend` | Top-5 por categoria, filtros e claims | `domain=recommendation` |
| `skill_faq` | FAQ e-commerce e revenda, só com evidência | `domain=faq` |

`RoutingDecision.plan` é o plano explícito de 1 passo. Replanejamento só em FAQ→handoff. Segurança **não** é skill — é pré-condição determinística.

In [4]:
from recfair.graphs.multiagent.skills import load_skill, skills_index

print(skills_index())
print("---")
print("skill_faq:", load_skill("skill_faq")["instruction"])
print("skill_recommend tools:", load_skill("skill_recommend")["tools"])


- skill_recommend: Top-5 por categoria, filtros e claims
- skill_faq: FAQ e-commerce e revenda, só com evidência
---
skill_faq: Responder políticas com os chunks recuperados. Sem catálogo. Sem evidência suficiente, sinalizar no_evidence para handoff.
skill_recommend tools: ['parse_intent', 'score_recommendation']


# G. Observabilidade

`AgentTrace` por nó: `agent_id`, `routing_reason`, LLM, tools, latência, tokens. Rota agregada `security > supervisor > …`. CLI `/trace` mostra o breakdown. O total do sistema **não** substitui a tabela por agente (célula H).

# H. Avaliação experimental por contexto

Cada contexto responde a uma pergunta diferente sobre o sistema. As métricas estão definidas no **glossário** abaixo e cada subseção (H.1 a H.4) traz:

1. **Resumo comparativo** — nota agregada por versão do sistema
2. **Tabela por caso** — resultado individual de cada pergunta de teste

| Seção | O que medimos | Versões comparadas | Casos |
| :--- | :--- | :--- | :--- |
| **H.1** | Qualidade do ranking Top-5 | E1 — Baseline × E2 — Workflow × E3 — Multi-agentes | 36 casos de catálogo |
| **H.2** | Segurança e guardrail | E1 — Baseline × E2 — Workflow × E3 — Multi-agentes | 5 casos de ataque e dados pessoais |
| **H.3** | Resposta em texto livre (FAQ) | Somente E3 — Multi-agentes | 10 perguntas de política/revenda |
| **H.4** | Destino do roteamento | Somente E3 — Multi-agentes | 10 perguntas (3 recomendação, 3 FAQ, 4 transbordo) |

Na célula de setup, `RUN_FAST=True` executa uma **amostra de 6 casos** (um por contexto, mais os três destinos de roteamento) para debug econômico. Use `RUN_FAST=False` para o relatório final.

Execute a célula de **execução** (carrega os três runs) e depois cada subseção H.1–H.4.

## H.0 — Glossário das métricas

Consulta rápida. Cada métrica abaixo está vinculada a um contexto de avaliação.

| Métrica | Contexto | O que mede |
| :--- | :--- | :--- |
| **Qualidade do ranking** | H.1 Recomendação | Proximidade da lista Top-5 ao gabarito, considerando a posição de cada produto |
| **Taxa de lista exata** | H.1 Recomendação | Percentual com os 5 produtos e ordem idênticos ao gabarito |
| **Taxa de erro grave** | H.1 Recomendação | Percentual de erros graves (categoria errada, produto inventado, etc.) |
| **Taxa de aprovação (segurança)** | H.2 Segurança | Percentual de casos de ataque/dados pessoais respondidos corretamente |
| **Similaridade semântica média** | H.3 FAQ | Média da similaridade por cosseno entre resposta gerada e referência |
| **Taxa de roteamento correto** | H.4 Roteamento | Percentual em que o supervisor encaminhou ao destino esperado |

**Limiar FAQ:** similaridade semântica ≥ 0,65 considera a resposta adequada.

In [5]:
display(HTML(render_metric_glossary()))


nome,contexto,o que mede,como interpretar
Qualidade do ranking (posição dos 5 produtos),Recomendação (H.1),"Quão perto a lista devolvida está do gabarito, considerando a posição de cada produto relevante entre os 5 slots. Valor 1,0 = ranking ideal; 0,0 = nenhum produto relevante na lista.","Quanto mais alto, melhor. Permite distinguir troca de ordem (erro leve) de lista completamente errada (erro grave)."
Taxa de lista exata,Recomendação (H.1),Percentual de casos em que os 5 produtos e a ordem coincidem exatamente com o gabarito.,Métrica mais rígida que a qualidade do ranking. Útil como referência legada.
Taxa de erro grave,Recomendação (H.1),"Percentual de casos classificados como erro grave: categoria errada, produto inventado, abstenção incorreta ou violação de guardrail.","Quanto mais baixo, melhor."
Taxa de aprovação (segurança),Segurança (H.2),Percentual de casos T26–T30 em que o ranking permanece correto e nenhum dado sensível ou marcador de ataque aparece na saída.,"Quanto mais alto, melhor. E1 e E2 tendem a falhar aqui."
Taxa de vazamento no guardrail,Segurança (H.2),"Percentual de casos em que padrões de dados pessoais, injeção ou jailbreak foram detectados na resposta (mesmo parcialmente).","Quanto mais baixo, melhor. Complementa a taxa de aprovação."
Similaridade semântica média,FAQ (H.3),Média da similaridade por cosseno entre o embedding da resposta gerada e o embedding da resposta de referência (modelo MiniLM multilíngue).,"Escala de 0 a 1. Acima de 0,65 consideramos semanticamente adequado. Perguntas complexas tendem a pontuar menos que perguntas diretas."
Taxa de aprovação (FAQ),FAQ (H.3),"Percentual de casos em que status=faq, similaridade ≥ limiar e nenhum produto (SKU) foi devolvido na resposta.","Quanto mais alto, melhor. Complementa a similaridade média."
Taxa de roteamento correto,Roteamento (H.4),"Percentual de casos em que o supervisor acionou o destino esperado: agente de recomendação, agente de FAQ, nó de transbordo humano ou redirecionamento fora de contexto.","Quanto mais alto, melhor. Medido apenas no E3 — Multi-agentes."
Distribuição por destino,Roteamento (H.4),"Taxa de acerto separada por destino: recomendação (3), FAQ (3), transbordo (2) e fora de contexto (4).",Revela se o supervisor erra mais em um tipo de pergunta.


In [6]:
manifests = {}
if not HAS_KEY:
    print(
        "Sem GOOGLE_API_KEY. Para o eval ponta a ponta:\n"
        "  1. cp .env.example .env  # preencha a chave\n"
        "  2. make data && make kernel\n"
        "  3. reexecute esta célula.\n"
        "Testes da régua (sem LLM): pytest tests/test_eval_metrics.py tests/test_guardrails.py"
    )
else:
    mode_label = "amostra rápida" if RUN_FAST else "golden-set completo"
    print(f"Executando {len(eval_cases)} casos ({mode_label})…")
    for arch in ("baseline", "workflow", "multiagent"):
        print(f"  run_eval({arch})…")
        manifests[arch] = run_eval(
            arch=arch,
            persist=not RUN_FAST,
            cases=eval_cases,
            fast_mode=RUN_FAST,
        )
        print(
            f"    {arch}: {manifests[arch]['run_id']} | "
            f"casos={manifests[arch]['cases_run']} | "
            f"golden={manifests[arch]['golden_revision']}"
        )
    if RUN_FAST:
        print()
        print("RUN_FAST=True — métricas são indicativas. Defina RUN_FAST=False para o relatório final.")


Unexpected argument 'thinking_level' provided to ChatGoogleGenerativeAI. Did you mean: 'thinking_budget'?


Executando 60 casos (golden-set completo)…
  run_eval(baseline)…


/home/andersonbr/estudos/unicamp-llm-agents/recfair/recfair/graphs/baseline.py:139: UserWarning: WARNING! thinking_level is not default parameter.
                thinking_level was transferred to model_kwargs.
                Please confirm that thinking_level is what you intended.
  structured = _get_structured_llm()


    baseline: c6d86c0d894f | casos=60 | golden=3cbcb3e4c4b9cec7
  run_eval(workflow)…
    workflow: 6d5cb78a9e25 | casos=60 | golden=3cbcb3e4c4b9cec7
  run_eval(multiagent)…


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

recommendation specialist failed: ValidationError: 1 validation error for RecFairOutput
items
  Value error, recommendation must have exactly 5 items [type=value_error, input_value=[RecommendationItem(sku='...e, fairness_notes=None)], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/andersonbr/estudos/unicamp-llm-agents/recfair/recfair/rag/faq_index.py:164: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='905d4b97-8a04-4bf4-a42e-3e0c80be7bd6', metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'FAQ de E-commerce O Boticário', 'source': '/home/andersonbr/estudos/unicamp-llm-agents/recfair/data/kb/faq.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3', 'tipo_fonte': 'pdf'}, page_content='logada\n \nno\n \nsite\n \nem\n \n"Minha\n \nConta"\n \n>\n \n"Meus\n \nPedidos"\n \n>\n \n"Solicitar\n \nDevolução"\n \nou\n \nentrando\n \nem\n \ncontato\n \ndireto\n \ncom\n \na\n \nCentral\n \nde\n \nRelacionamento.\n \nPergunta\n \n10:\n \nQuais\n \nsão\n \nas\n \ndiretrizes\n \nde\n \nembalagem,\n \nnota\n \nfiscal\n \ne\n \npostagem\n \nnos\n \nCorreios\n \npara\n \na\n \ndevolução\n \nde\n \nprodutos?\n \nResposta\n:\n \nApós\n \na\n \nabertura\n \nda\n \nsolicitação\n \nde\n \ndevoluç

    multiagent: ae3f3348d3e4 | casos=60 | golden=3cbcb3e4c4b9cec7


### H.1 — Recomendação de produtos

**Pergunta avaliada:** A lista Top-5 devolvida está próxima do gabarito do motor de ranking?

**Comparação:** E1 — Baseline × E2 — Workflow × E3 — Multi-agentes

**Métrica principal:** Qualidade do ranking (posição dos 5 produtos)

**Casos:** T01–T25, T31–T38 e T61–T63 (36 perguntas de catálogo)

In [7]:
if not manifests:
    print("Execute a célula de execução acima para carregar os resultados.")
else:
    ma = manifests.get("multiagent")
    display(
        HTML(
            render_context_evaluation_section(
                manifests,
                ma["records"],
                "recomendacao",
                output_column="E3 — Multi-agentes",
            )
        )
    )


### H.2 — Segurança e guardrail

**Pergunta avaliada:** O sistema responde corretamente sem vazar dados sensíveis nem cair em ataques?

**Comparação:** E1 — Baseline × E2 — Workflow × E3 — Multi-agentes

**Métrica principal:** Taxa de aprovação nos casos de segurança

**Casos:** T26–T30 (dados pessoais, injeção de prompt e jailbreak)

In [8]:
if not manifests:
    print("Execute a célula de execução acima para carregar os resultados.")
else:
    ma = manifests.get("multiagent")
    display(
        HTML(
            render_context_evaluation_section(
                manifests,
                ma["records"],
                "seguranca",
                output_column="E3 — Multi-agentes",
            )
        )
    )


### H.3 — Perguntas frequentes

**Pergunta avaliada:** A resposta em texto livre é semanticamente próxima da referência na base de conhecimento?

**Comparação:** Somente E3 — Multi-agentes (única com agente de FAQ)

**Métrica principal:** Similaridade semântica média (cosseno entre embeddings)

**Casos:** T39–T40 e T44–T51 (10 perguntas de política e revenda)

In [9]:
if not manifests:
    print("Execute a célula de execução acima para carregar os resultados.")
else:
    ma = manifests.get("multiagent")
    display(
        HTML(
            render_context_evaluation_section(
                manifests,
                ma["records"],
                "faq",
                output_column="E3 — Multi-agentes",
            )
        )
    )


### H.4 — Roteamento do supervisor

**Pergunta avaliada:** O supervisor encaminhou para o destino correto: recomendação, FAQ ou transbordo humano?

**Comparação:** Somente E3 — Multi-agentes (única com supervisor)

**Métrica principal:** Taxa de roteamento correto

**Casos:** T41–T43 e T52–T58 (10 perguntas: 3 recomendação, 3 FAQ, 4 transbordo)

In [10]:
if not manifests:
    print("Execute a célula de execução acima para carregar os resultados.")
else:
    ma = manifests.get("multiagent")
    display(
        HTML(
            render_context_evaluation_section(
                manifests,
                ma["records"],
                "roteamento",
                output_column="E3 — Multi-agentes",
            )
        )
    )


versão,casos avaliados,taxa de roteamento correto,acerto em recomendação,acerto em FAQ,acerto em transbordo,acerto em fora de contexto
E3 — Multi-agentes,12,83.3%,100.0%,66.7%,50.0%,100.0%
caso,status final,pergunta,E3 — Multi-agentes,destino esperado,rota executada,status final
T41,erro*,Qual o prazo de entrega dos pedidos?,—,perguntas frequentes,security > supervisor > faq > handoff,handoff
T42,sucesso,Quais os produtos de cabelo mais vendidos?,F3P9W2 → A8T3K5 → 3G7P2W → H8Q3N1 → L6K1C8,recomendação,security > supervisor > recommendation,recommendation
T43,sucesso,Como faço para declarar o Imposto de Renda da Receita Federal?,—,fora de contexto,security > supervisor > out_of_context,out_of_context
T52,sucesso,Me mostra as colônias masculinas Malbec mais vendidas.,Q4H8L2 → 7K2N9A → 3R1B6M → 5J8P2X → W9C5TD,recomendação,security > supervisor > recommendation,recommendation
T53,sucesso,Quais perfumes femininos mais vendidos na última semana?,G7Q2D4 → 8K2F6Q → 6P8H3A → 1M9T5B → 5X3R8K,recomendação,security > supervisor > recommendation,recommendation
T54,sucesso,É aceito boleto bancário como forma de pagamento no site?,—,perguntas frequentes,security > supervisor > faq,faq
T55,sucesso,Como funciona a troca de produtos comprados online?,—,perguntas frequentes,security > supervisor > faq,faq
T56,erro*,Como faço o cadastro para ser revendedor do Boticário?,—,transbordo,security > supervisor > faq,faq


### H.5 — Instrumentação complementar

Requisitos funcionais (referência legada) e custo por agente no E3 — Multi-agentes.

In [11]:
if manifests:
    ma = manifests["multiagent"]
    display(HTML(render_rf_breakdown_table(ma["resumo_v3"], title="Requisitos funcionais (referência)")))
    display(HTML(render_agent_cost_table(ma["records"], title="Custo e latência por agente (E3 — Multi-agentes)")))


requisito,conceito,passou,total,taxa
RF-01,Somente SKUs do catálogo — nenhum código de produto inventado.,34,34,1.0
RF-02,Filtros de categoria e marca da pergunta respeitados na lista Top-5.,33,33,1.0
RF-03,Lista com exatamente 5 SKUs na mesma ordem do gabarito de ranking.,24,35,0.6857
RF-04,Abstenção quando a categoria não está clara ou não foi informada.,2,3,0.6667
RF-05,Abstenção para marca ou categoria inexistente no catálogo.,3,3,1.0
RF-06,SKUs proibidos (janela de fairness) ausentes da recomendação.,1,2,0.5
RF-07,Pelo menos duas marcas distintas quando a pergunta exige diversidade.,19,20,0.95


agente,casos,latencia s,chamadas llm,tool calls,tokens entrada,tokens saida,custo usd
faq,13,1.32,1.0,1.0,2506,82,0.000957
handoff,6,0.0,0.0,0.0,0,0,0.0
out_of_context,4,0.0,0.0,0.0,0,0,0.0
recommendation,44,0.99,1.0,6.75,403,50,0.000246
security,63,0.0,0.0,0.0,0,0,0.0
supervisor,63,0.9,1.0,0.0,534,84,0.000369


# I. Novos modos de falha

Modos introduzidos pela arquitetura multiagente, verificados no run completo (`golden_revision=3cbcb3e4c4b9cec7`, modelo `gemini-3.5-flash-lite`, 60 casos). A rota de cada caso está na coluna *rota* das tabelas H.1–H.4.

| Modo de falha | Ocorreu? | Caso e o que o trace mostrou |
| :--- | :---: | :--- |
| **Roteamento para o agente errado** | **Sim (2/12)** | **T56** (*cadastro revendedor*): supervisor→`faq`, esperado `handoff` (H.4: acerto transbordo 50%). Trace: `security > supervisor > faq`. **T41** (*prazo de entrega*): rota `security > supervisor > faq > handoff` — FAQ não encontrou evidência suficiente e replanejou; gabarito esperava resposta FAQ direta. |
| **Agente respondendo fora do escopo** | **Não observado** | Nos 10 casos FAQ aprovados e nas 44 recomendações, nenhum retornou SKUs quando `status=faq` nem `answer_text` de política quando `status=recommendation`. O contrato `AgentResult` + verify `not skus` impediu mistura visível. |
| **Pergunta composta atendida por um só agente** | **Sim (latente)** | **T31** (memória 2 turnos): rota correta `recommendation`, mas nDCG=0,8304 — o supervisor não decompõe turnos; falha herdada do `parse_intent` E2, não de roteamento. |
| **Achado correto perdido na agregação** | **Não** | FAQ transporta chunks em `AgentResult.evidence`; recomendação preserva `ScoreTrace`. Nenhum caso em que o destino final precisou reler a query bruta para inferir o que o upstream produziu. |
| **Laço de transferências sem convergência** | **Não** | `recursion_limit=16`; único replanejamento permitido FAQ→`handoff`. Run completo: 6 handoffs, 0 `halt_reason=recursion_limit`. T44/T46/T47 convergiram em handoff após `no_evidence`. |
| **Plano que ignora parte da pergunta** | **Parcial (engine, não supervisor)** | **T20** (preço ≤ R$60): nDCG=0,6164 — intent/filter no pipeline E2, rota correta. **T17** (colônia ≤ R$180): erro grave — mesma causa. O plano de 1 passo do supervisor não corrige constraints numéricas. |
| **Perda/corrupção de contexto entre agentes** | **Parcial** | `security_node` redact PII **antes** do supervisor (T26–T30: vazamento na saída = False em E2 e E3). Risco latente: redact agressivo poderia truncar intent — não medido neste golden-set. |
| **FAQ além da evidência / handoff prematuro** | **Sim (4/10 FAQ)** | **T44, T46, T47**: `retrieve_faq` sem chunk útil → `no_evidence` → handoff (similaridade 0,15 / 0,27 / −0,03). **T51**: respondeu FAQ (0,647) mas abaixo do limiar 0,65. Trace FAQ: ~1,32 s e ~2 506 tokens entrada por chamada — etapa mais cara quando acionada. |
| **Claims semânticos insuficientes** | **Sim (herdado + parcial)** | **T21/T22**: nDCG 0,87 / 0,79 — `match_claims_hybrid` não recuperou SKU-alvo (`V2L9D6`). **T18/T24/T37**: permutações/leves (nDCG=1,0, exact fail). Mesmo pipeline E2 encapsulado; fallback embed não fechou o gap. |
| **Outro: acerto “falso” de guardrail no E2** | **Sim (métrica)** | H.2: E2 e E3 ambos 100% aprovação T26–T30 porque a régua mede **vazamento na saída**, não sanitização na entrada. O ganho real do E3 é arquitetural (`security_node` impede PII/injection de chegar ao LLM) — ver testes `tests/test_guardrails.py`. |

**Modos da régua (não são falhas do agente):** T12 vs T14 (nDCG distingue permutação de erro grave); `false_positive_gap`; tightening de `G_need` no legado — o E3 lê `gap_intentional_pass` e nDCG@5 (ADR 0004).

# J. Análise arquitetural

Run desta sessão: três arquiteturas × 60 casos, mesmo modelo `gemini-3.5-flash-lite`, `golden_revision=3cbcb3e4c4b9cec7`. Manifests: `baseline:c6d86c0d894f`, `workflow:6d5cb78a9e25`, `multiagent:ae3f3348d3e4`.

## 1. Limitações da v2 que a divisão resolveu (com evidência)

| Limitação E2 | Evidência no trace / H |
| :--- | :--- |
| **Domínio único** (sem FAQ/revenda/transbordo/fora de contexto) | **H.3:** 60% aprovação FAQ (6/10), similaridade média 0,589 — capacidade inexistente no E2. **H.4:** 83,3% roteamento (10/12); `out_of_context` 100% (T43, T57–T59). Nós `handoff`/`out_of_context` atendem sem LLM (0 tokens). |
| **Sanitização na entrada** | `security_node` em **todos** os 63 casos E3 (tabela H.5: 0 LLM). T26–T30: ranking correto + vazamento=False; PII/injection não chegam aos agentes LLM. |
| **Claims por substring** (T18/T21/T22) | Matcher híbrido injetado no especialista; melhora parcial (T18 nDCG=1,0), mas T21/T22 ainda falham — limite do embed, não do supervisor. |

## 2. Limitações que permaneceram ou surgiram

**Permaneceram (mesmo engine E2 dentro do especialista):** 
- intent/preço (**T17, T20**),
- memória (**T09, T31**)
- abstenção (**T36**)
- desempate (**T12, T24, T37**). 

**H.1:** nDCG@5 E2 **0,9779** vs E3 **0,9641** (−1,4 pp); taxa lista exata **84,9%** vs **66,7%** (−18,2 pp); taxa erro grave igual (**9,1%**). 

O escopo estrito de recomendação continua **melhor no E2** — mais simples e, nesta régua, mais acurado na ordem exata.

**Surgiram com multiagente:**

- **Custo e latência:** por caso de recomendação (H.1 instrumentação): 
    - E2 **0,85 s**, **1,09** chamadas LLM, **$0,0003**
    - E3 **2,14 s**, **2,18** LLM, **$0,0007**. 
Overhead do supervisor (+0,9 s, +1 LLM) antes do especialista.

- **Custo por agente (H.5, média quando acionado):** 
    - **FAQ 1,32 s** e **2 506** tokens entrada (dominante quando roteado); 
    - **recomendação 0,99 s**; **supervisor 0,9 s**; 
    - `security`/`handoff`/`out_of_context` = 0 LLM — confirmando que FAQ + roteamento explicam o delta, não o ranking.
- **Risco de rota errada:** T56 (revenda→FAQ em vez de handoff); ambiguidade T41 (FAQ→handoff).
- **Dependência operacional:** FAISS FAQ + MiniLM (`make data`, `HF_TOKEN`); índices não versionados no git.

## 3. Agente concentrou custo e latência — o que fazer

| Agente/nó | Latência média | O que faria |
| :--- | ---: | :--- |
| **FAQ** | 1,32 s | Melhorar chunking/PDF (`faq.pdf` quebras ruins nos warnings do run); re-rank; elevar k ou híbrido BM25; regra “revenda cadastral → handoff” antes do LLM (corrige T56/T47). |
| **Supervisor** | 0,9 s | Pré-roteamento regex para padrões de handoff/revenda; reduzir prompt (só índice de skills). |
| **Recomendação** | 0,99 s | Manter engine E2; investir em intent/preço, não em quarto agente. |

## 4. Algum agente poderia voltar a ser etapa de fluxo?

- **`security`, `handoff`, `out_of_context`:** já são nós determinísticos — decisão correta (economia de tokens, ADR 0003/0005).
- **Supervisor:** não — a escolha FAQ vs ranking vs transbordo depende da entrada; pipeline fixo não cobre T39–T58.
- **FAQ:** justificado enquanto H.3 > 0 e domínio distinto (RAG + síntese ground-only). Sem FAQ medido, voltaria a handoff genérico.
- **Recomendação:** conceitualmente é o workflow E2 encapsulado; mantê-lo como **agente** isola tools/prompt de catálogo do FAQ e permite avaliação H.1 isolada.

## 5. A hipótese da seção B se confirmou?

**Parcialmente.** Confirmado: ampliar escopo (FAQ + roteamento + transbordo + fora de contexto) com supervisor; guardrail na entrada; skills sob demanda. 

**Não confirmado** para ranking: E3 **piorou** lista exata e nDCG vs E2 no escopo H.1, em linha com a hipótese explícita de “empate ou queda aceitável”. Claims semânticos e FAQ ainda precisam iterar (T21/T22, T44/T46/T47).

---

### Pergunta obrigatória

> **Qual das versões construídas até aqui você levaria para uso real, e que evidência ainda falta para sustentar essa escolha?**

**Escolha: E3 (`multiagent`), com ressalvas no escopo de recomendação.**

- **Por quê E3 e não E2 puro:** o produto-alvo não é só Top-5 de catálogo — precisa responder políticas (H.3), recusar fora de domínio (H.4 100% em `out_of_context`) e transbordar cadastro/revenda (handoff). Nenhuma dessas capacidades existe no workflow. Os nós `handoff` e `out_of_context` são **essenciais** para economia (zero LLM) e UX previsível.
- **Por quê não baseline:** H.1 nDCG 0,74 vs 0,96+ nas versões com engine.
- **Ressalva explícita:** para **só** recomendação Top-5, o **E2 permanece superior** neste run (nDCG 0,9779, exact 84,9%). Em produção híbrida, manteria o grafo E3 mas priorizaria melhorias no especialista de recomendação (intent/preço/claims) **antes** de adicionar agentes ou MCP.

**Evidência que ainda falta (E4):**

1. Regressão A/B com tráfego real ou amostra > 60 casos, separando sessões “só catálogo” vs “FAQ/política”.
2. Métricas de **falso positivo** do guardrail (query legítima com CPF na tarefa).
3. Latência p95 por rota (FAQ vs recomendação) sob carga; custo mensal com volume de FAQ.
4. Fechar gap H.1 (meta: nDCG E3 ≥ E2) via intent — **sem** remover supervisor/FAQ.
5. Manifests persistidos em `eval/runs/` (gitignored) para auditoria externa além do notebook.

**Ponto de partida E4:** regras determinísticas pré-supervisor para handoff/revenda (T56); chunking FAQ; só então reconsiderar quarto agente ou MCP.
